In [1]:
from langgraph.graph import StateGraph ,START ,END
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

d:\Lakshya\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
model =  ChatOllama(model = 'minimax-m3:cloud', temperature=0.5)

# define the Class State
class chatState(TypedDict):
    # define the properties
    messages : Annotated[list[BaseMessage], add_messages]

# define the methord
def chatNode(State: chatState):
    messages = State['messages']
    response = model.invoke(messages)
    return {'messages':[response]}

In [3]:
graph = StateGraph(chatState)

# add node
graph.add_node('chat_node', chatNode)

# add edge
graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

conn = sqlite3.connect(database = "chatbot.db",check_same_thread= False)
# define the checkpointer
checkpointer = SqliteSaver(conn= conn) 
chatbot = graph.compile(checkpointer= checkpointer)

config = {'configurable':{"thread_id":'1'}}
result = chatbot.stream({'messages':[ HumanMessage(content = "Hii, How are you")]}, config= config , stream_mode="messages")
print(result)

<generator object Pregel.stream at 0x00000214E75AB850>
